# Lesson 6.2 - Data & Feature Pipelines (toy pipeline demo)

This notebook simulates a production-style batch feature pipeline: raw input, quality checks, feature engineering, and curated output for model training.

## Objectives

- Build a reproducible batch pipeline with explicit stages.
- Add lightweight data-quality checks.
- Create training-ready features with leakage-aware transformations.
- Save curated outputs and pipeline metadata.

## Setup

In [1]:
from __future__ import annotations

import json
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd

SEED = 42
rng = np.random.default_rng(SEED)

RAW_DIR = Path("artifacts/lesson6_2/raw")
CURATED_DIR = Path("artifacts/lesson6_2/curated")
RAW_DIR.mkdir(parents=True, exist_ok=True)
CURATED_DIR.mkdir(parents=True, exist_ok=True)

## Section A - Simulate Raw Batch Data

In [2]:
n = 2000
base_date = pd.Timestamp("2026-01-01")

raw_df = pd.DataFrame(
    {
        "customer_id": np.arange(1, n + 1),
        "snapshot_date": base_date + pd.to_timedelta(rng.integers(0, 90, size=n), unit="D"),
        "tenure_days": rng.integers(1, 1200, size=n),
        "monthly_spend": np.clip(rng.normal(45, 15, size=n), 2, None),
        "support_tickets_30d": rng.poisson(1.2, size=n),
        "last_login_days_ago": rng.integers(0, 90, size=n),
        "region": rng.choice(["north", "south", "east", "west"], size=n),
    }
)

raw_df["churn_label"] = (
    (raw_df["last_login_days_ago"] > 40).astype(int)
    | (raw_df["support_tickets_30d"] > 2).astype(int)
)

raw_path = RAW_DIR / "customer_snapshot.csv"
raw_df.to_csv(raw_path, index=False)

raw_df.head()

,customer_id,snapshot_date,tenure_days,monthly_spend,support_tickets_30d,last_login_days_ago,region,churn_label
0,1,2026-01-09,1053,29.632066,3,12,east,1
1,2,2026-03-11,75,58.620943,3,45,south,1
2,3,2026-02-28,477,26.959551,3,32,north,1
3,4,2026-02-09,550,40.704508,1,57,north,1
4,5,2026-02-08,957,36.209225,1,57,east,1


## Section B - Data Quality Checks

In [3]:
def run_quality_checks(df: pd.DataFrame) -> dict:
    checks = {}
    checks["null_ratio_monthly_spend"] = float(df["monthly_spend"].isna().mean())
    checks["null_ratio_region"] = float(df["region"].isna().mean())
    checks["invalid_tenure_rows"] = int((df["tenure_days"] < 0).sum())
    checks["invalid_login_rows"] = int((df["last_login_days_ago"] < 0).sum())
    checks["duplicate_customer_snapshot"] = int(df.duplicated(subset=["customer_id", "snapshot_date"]).sum())
    checks["row_count"] = int(len(df))
    checks["status"] = "pass"

    if (
        checks["null_ratio_monthly_spend"] > 0.02
        or checks["invalid_tenure_rows"] > 0
        or checks["duplicate_customer_snapshot"] > 0
    ):
        checks["status"] = "fail"

    return checks


quality_report = run_quality_checks(raw_df)
quality_report

{'null_ratio_monthly_spend': 0.0,
 'null_ratio_region': 0.0,
 'invalid_tenure_rows': 0,
 'invalid_login_rows': 0,
 'duplicate_customer_snapshot': 0,
 'row_count': 2000,
 'status': 'pass'}

## Section C - Feature Engineering Pipeline

In [4]:
def build_features(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()

    out["spend_per_ticket_30d"] = out["monthly_spend"] / (1 + out["support_tickets_30d"])
    out["is_dormant_30d"] = (out["last_login_days_ago"] >= 30).astype(int)
    out["tenure_bucket"] = pd.cut(
        out["tenure_days"],
        bins=[0, 90, 365, 730, 99999],
        labels=["new", "growing", "established", "long_term"],
    )

    out = pd.get_dummies(out, columns=["region", "tenure_bucket"], drop_first=False)
    return out


features_df = build_features(raw_df)
features_df.head()

,customer_id,snapshot_date,tenure_days,monthly_spend,support_tickets_30d,last_login_days_ago,churn_label,spend_per_ticket_30d,is_dormant_30d,region_east,region_north,region_south,region_west,tenure_bucket_new,tenure_bucket_growing,tenure_bucket_established,tenure_bucket_long_term
0,1,2026-01-09,1053,29.632066,3,12,1,7.408017,0,True,False,False,False,False,False,False,True
1,2,2026-03-11,75,58.620943,3,45,1,14.655236,1,False,False,True,False,True,False,False,False
2,3,2026-02-28,477,26.959551,3,32,1,6.739888,1,False,True,False,False,False,False,True,False
3,4,2026-02-09,550,40.704508,1,57,1,20.352254,1,False,True,False,False,False,False,True,False
4,5,2026-02-08,957,36.209225,1,57,1,18.104613,1,True,False,False,False,False,False,False,True


## Section D - Persist Curated Dataset + Metadata

In [5]:
curated_path = CURATED_DIR / "customer_features_batch.csv"
meta_path = CURATED_DIR / "pipeline_metadata.json"

features_df.to_csv(curated_path, index=False)

metadata = {
    "run_time_utc": datetime.utcnow().isoformat() + "Z",
    "input_path": str(raw_path),
    "output_path": str(curated_path),
    "row_count": int(len(features_df)),
    "feature_count": int(features_df.shape[1]),
    "quality_report": quality_report,
}
meta_path.write_text(json.dumps(metadata, indent=2), encoding="utf-8")

metadata

/tmp/ipykernel_576277/1672144606.py:7: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "run_time_utc": datetime.utcnow().isoformat() + "Z",


{'run_time_utc': '2026-07-03T05:45:06.222159Z',
 'input_path': 'artifacts/lesson6_2/raw/customer_snapshot.csv',
 'output_path': 'artifacts/lesson6_2/curated/customer_features_batch.csv',
 'row_count': 2000,
 'feature_count': 17,
 'quality_report': {'null_ratio_monthly_spend': 0.0,
  'null_ratio_region': 0.0,
  'invalid_tenure_rows': 0,
  'invalid_login_rows': 0,
  'duplicate_customer_snapshot': 0,
  'row_count': 2000,
  'status': 'pass'}}

## Optional - Airflow DAG Pattern (Pseudo-code)

This cell prints a minimal DAG skeleton to show how the same stages are orchestrated in production schedulers.

In [6]:
airflow_dag_example = '''
from airflow import DAG
from airflow.operators.python import PythonOperator

with DAG("daily_feature_pipeline", schedule="0 2 * * *", start_date=..., catchup=False) as dag:
    ingest = PythonOperator(task_id="ingest_raw", python_callable=ingest_raw)
    validate = PythonOperator(task_id="validate_data", python_callable=validate_data)
    transform = PythonOperator(task_id="build_features", python_callable=build_features)
    publish = PythonOperator(task_id="publish_curated", python_callable=publish_data)

    ingest >> validate >> transform >> publish
'''

print(airflow_dag_example)


from airflow import DAG
from airflow.operators.python import PythonOperator

with DAG("daily_feature_pipeline", schedule="0 2 * * *", start_date=..., catchup=False) as dag:
    ingest = PythonOperator(task_id="ingest_raw", python_callable=ingest_raw)
    validate = PythonOperator(task_id="validate_data", python_callable=validate_data)
    transform = PythonOperator(task_id="build_features", python_callable=build_features)
    publish = PythonOperator(task_id="publish_curated", python_callable=publish_data)

    ingest >> validate >> transform >> publish



## Connect to Theory

- Section A maps to ingestion layer.
- Section B enforces data quality contracts.
- Section C defines reusable feature logic.
- Section D persists artifacts and metadata for reproducibility.

In production, the same pattern extends with backfills, lineage, SLA dashboards, and feature store materialization.

## Business Case Studies & Exceptions

### Case 1: Churn prediction with daily batch features

- **Pattern**: run nightly ETL + feature pipeline, score users each morning.
- **Benefit**: predictable costs and simple operations.
- **Exception**: campaign-time interventions may need intraday refreshes.

### Case 2: Real-time recommendation systems

- **Pattern**: combine batch historical features with streaming session features.
- **Benefit**: balances stable historical context with fresh intent signals.
- **Exception**: streaming systems add operational complexity (late events, watermarking, on-call burden).

### Case 3: Feature store adoption decision

- **Rule of thumb**: adopt a full feature store when feature reuse and online/offline consistency pain exceed current pipeline overhead.

## Interview Questions & Answers

1. **Q: What is the role of data pipelines in ML systems?**
   **A:** They deliver validated, timely, and reproducible datasets/features for training and inference.

2. **Q: Batch vs streaming in one line?**
   **A:** Batch is scheduled and simpler; streaming is continuous and lower-latency but harder to operate.

3. **Q: What is point-in-time correctness?**
   **A:** Training features must only use information available at prediction time.

4. **Q: What is training-serving skew?**
   **A:** Mismatch between feature computation in model training and production inference.

5. **Q: Why are data quality checks non-negotiable?**
   **A:** They prevent silent corruption propagating into model decisions.

6. **Q: What belongs in pipeline metadata?**
   **A:** input/output paths, run timestamps, quality checks, schema version, and code version.

7. **Q: When does a team need a feature store?**
   **A:** When many models reuse features and online/offline consistency becomes painful.

8. **Q: What causes schema drift incidents?**
   **A:** Upstream schema changes without contract enforcement or compatibility management.

9. **Q: How do you debug a sudden feature distribution shift?**
   **A:** Trace upstream source changes, transformation logic, and ingestion delays by run ID.

10. **Q: Why keep raw snapshots?**
   **A:** For reproducibility, root-cause analysis, and historical backfills.

11. **Q: What is the first reliability improvement for fragile pipelines?**
   **A:** Add explicit quality checks and fail-fast behavior before modeling improvements.

12. **Q: How do you decide refresh cadence?**
   **A:** Based on business decision latency requirements, data arrival patterns, and operating cost.
